In [4]:
!pip install PyPDF2 nltk scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.5 MB/s eta 0:00:00


In [5]:
from google.colab import files
uploaded = files.upload()

Saving sample_resume.pdf to sample_resume (1).pdf


In [16]:
import PyPDF2

def extract_text(pdf_path):
    text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            if page.extract_text():
                text += page.extract_text()
    return text

# Get uploaded file name
for file_name in uploaded.keys():
    resume_text = extract_text(file_name)
    print(f"\nProcessing: {file_name}")


Processing: sample_resume (1).pdf


In [8]:
job_descriptions = {
    "Data Scientist": """
    python machine learning deep learning statistics data analysis pandas numpy scikit-learn
    data visualization matplotlib seaborn feature engineering model evaluation regression classification
    sql big data data cleaning hypothesis testing
    """,

    "Web Developer": """
    html css javascript react node.js express mongodb frontend backend full stack development
    rest api responsive design web application git github debugging
    """,

    "ML Engineer": """
    python machine learning deep learning tensorflow pytorch nlp computer vision
    model deployment docker flask api cloud computing data pipelines automation
    """,

    "Data Analyst": """
    python sql excel data analysis data visualization pandas numpy tableau power bi
    dashboards reporting business analysis data cleaning statistics
    """,

    "Software Engineer": """
    java python c++ data structures algorithms problem solving oops system design
    software development debugging git github databases operating systems
    """
}

In [20]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    words = [w for w in text.split() if w not in ENGLISH_STOP_WORDS]
    return " ".join(words)

# Apply preprocessing
resume_clean = preprocess(resume_text)

# Clean job descriptions too
job_clean = {k: preprocess(v) for k, v in job_descriptions.items()}

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [resume_clean] + list(job_clean.values())

vectorizer = TfidfVectorizer(ngram_range=(1,2))
tfidf_matrix = vectorizer.fit_transform(documents)

In [22]:
feature_names = vectorizer.get_feature_names_out()
resume_vector = tfidf_matrix[0].toarray()[0]

top_indices = resume_vector.argsort()[-10:][::-1]

print("\n Top Resume Keywords:")
for i in top_indices:
    print(feature_names[i])


 Top Resume Keywords:
using
using nlp
sharma skills
sharma
tech cse
tech
using linear
skills
prediction using
projects house


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

resume_vector = tfidf_matrix[0]

scores = {}
for i, role in enumerate(job_descriptions.keys()):
    job_vector = tfidf_matrix[i+1]
    score = cosine_similarity(resume_vector, job_vector)[0][0]
    scores[role] = score

In [26]:
# Sort roles by score (highest first)
sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)

print("="*40)
print(" RESUME ANALYSIS RESULT")
print("="*40)

# Best role
print(f"\n Best Role: {sorted_scores[0][0]}")

# Ranking
print("\n Role Ranking:")
for i, (role, score) in enumerate(sorted_scores, start=1):
    print(f"{i}. {role} → {round(score*100,2)}%")
    confidence = sorted_scores[0][1]

if confidence > 0.7:
    level = "High"
elif confidence > 0.4:
    level = "Medium"
else:
    level = "Low"

print(f"\n📈 Confidence Level: {level}")

# Skill gap
best_role = sorted_scores[0][0]

resume_words = set(resume_text.lower().split())
job_words = set(job_descriptions[best_role].lower().split())

missing_skills = job_words - resume_words

print("\n Missing Skills:")
print(", ".join(missing_skills))

 RESUME ANALYSIS RESULT

 Best Role: Data Scientist

 Role Ranking:
1. Data Scientist → 26.43%
2. Data Analyst → 15.67%
3. ML Engineer → 13.31%
4. Software Engineer → 2.9%
5. Web Developer → 0.0%

📈 Confidence Level: Low

 Missing Skills:
pandas, visualization, learning, statistics, matplotlib, deep, testing, scikit-learn, numpy, feature, engineering, classification, evaluation, sql, cleaning, seaborn, model, big, python, hypothesis


In [23]:
resume_words = set(resume_clean.split())
job_words = set(job_clean[best_role].split())

missing_skills = job_words - resume_words
missing_skills = list(missing_skills)[:5]

print("\n Missing Skills (Top 5):")
print(", ".join(missing_skills))


 Missing Skills (Top 5):
feature, engineering, model, deep, classification
